# 🎵 EAS 587 – Spotify Analysis | Silver Layer
**Team TriForce** — Dilip, Harsha, Pavan

Reads Bronze, applies Phase 1 cleaning, engineers `popularity_tier`,
writes a managed Silver Delta table — no DBFS paths.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Configuration ──────────────────────────────────────────────────
BRONZE_TABLE = "workspace.spotify_bronze.raw_tracks"
SILVER_CAT   = "workspace"
SILVER_SCH   = "spotify_silver"
SILVER_TBL   = "clean_tracks"

AUDIO_FEATURES = [
    "danceability", "energy", "valence",
    "acousticness", "instrumentalness",
    "tempo", "loudness", "speechiness"
]

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_CAT}.{SILVER_SCH}")
print(f"✅ Schema '{SILVER_CAT}.{SILVER_SCH}' ready")

✅ Schema 'workspace.spotify_silver' ready


In [0]:
# Read Bronze
df_bronze = spark.table(BRONZE_TABLE)
print(f"📥 Bronze rows : {df_bronze.count():,}")

📥 Bronze rows : 114,000


In [0]:
# 1. Select & cast columns we need
df = df_bronze.select(
    F.col("track_id"),
    F.col("track_name"),
    F.col("artists"),
    F.col("album_name"),
    F.col("track_genre").alias("genre"),
    F.col("popularity").cast("int"),
    F.col("duration_ms").cast("long"),
    F.col("explicit").cast("boolean"),
    *[F.col(c).cast("double") for c in AUDIO_FEATURES]
)

In [0]:
# 2. Drop nulls in key columns
key_cols = ["track_id", "popularity"] + AUDIO_FEATURES
df_clean = df.dropna(subset=key_cols)

In [0]:
# 3. Remove duplicate track_ids — keep first occurrence
w = Window.partitionBy("track_id").orderBy(F.monotonically_increasing_id())
df_clean = (
    df_clean
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

In [0]:
# 4. Remove tracks with popularity == 0
df_clean = df_clean.filter(F.col("popularity") > 0)

In [0]:
# 5. Remove out-of-range audio feature values
range_filters = (
    (F.col("danceability")     .between(0.0, 1.0)) &
    (F.col("energy")           .between(0.0, 1.0)) &
    (F.col("valence")          .between(0.0, 1.0)) &
    (F.col("acousticness")     .between(0.0, 1.0)) &
    (F.col("instrumentalness") .between(0.0, 1.0)) &
    (F.col("speechiness")      .between(0.0, 1.0)) &
    (F.col("tempo")            > 0) &
    (F.col("loudness")         .between(-80.0, 5.0))
)
df_clean = df_clean.filter(range_filters)
print(f"🧹 Rows after cleaning : {df_clean.count():,}")

🧹 Rows after cleaning : 80,140


In [0]:
# 6. Engineer popularity_tier — matches Phase 1 thresholds
df_silver = df_clean.withColumn(
    "popularity_tier",
    F.when(F.col("popularity") < 30, F.lit("Low"))
     .when(F.col("popularity") < 60, F.lit("Medium"))
     .otherwise(F.lit("High"))
)

In [0]:
# Write managed Silver Delta table
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_CAT}.{SILVER_SCH}.{SILVER_TBL}")
)

print(f"✅ Silver table '{SILVER_CAT}.{SILVER_SCH}.{SILVER_TBL}' written")

✅ Silver table 'workspace.spotify_silver.clean_tracks' written


## ✅ Validation — Silver table exists

In [0]:
%sql
SHOW TABLES IN workspace.spotify_silver;

database,tableName,isTemporary
spotify_silver,clean_tracks,false


In [0]:
%sql
SELECT * FROM workspace.spotify_silver.clean_tracks LIMIT 5;

track_id,track_name,artists,album_name,genre,popularity,duration_ms,explicit,danceability,energy,valence,acousticness,instrumentalness,tempo,loudness,speechiness,popularity_tier
0000vdREvCVMxbQTkS888c,Lolly,Rill,Lolly,german,44,160725,true,0.91,0.374,0.432,0.0757,0.00301,104.042,-9.844,0.199,Medium
000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now (Glee Cast Version),Glee Cast,Glee Love Songs,club,47,322933,false,0.269,0.516,0.341,0.406,0.0,178.174,-7.361,0.0366,Medium
000Iz0K615UepwSJ5z2RE5,Böxig Leise - Pig & Dan Remix,Paul Kalkbrenner;Pig&Dan,X,minimal-techno,22,515360,false,0.686,0.56,0.108,0.00114,0.181,119.997,-13.264,0.0462,Low
000RDCYioLteXcutOjeweY,Teeje Week,Jordan Sandhu,Teeje Week,hip-hop,62,190203,false,0.679,0.77,0.839,0.0583,0.0,161.721,-3.537,0.19,High
000qpdoc97IMTBvF8gwcpy,Tief,Paul Kalkbrenner,Zeit,minimal-techno,19,331240,false,0.519,0.431,0.234,9.64E-4,0.72,129.971,-13.606,0.0291,Low


In [0]:
%sql
SELECT
    popularity_tier,
    COUNT(*)                      AS track_count,
    ROUND(AVG(popularity),   2)   AS avg_popularity,
    ROUND(AVG(danceability), 3)   AS avg_danceability,
    ROUND(AVG(energy),       3)   AS avg_energy,
    ROUND(AVG(valence),      3)   AS avg_valence
FROM workspace.spotify_silver.clean_tracks
GROUP BY popularity_tier
ORDER BY popularity_tier;

popularity_tier,track_count,avg_popularity,avg_danceability,avg_energy,avg_valence
High,9800,67.53,0.595,0.638,0.485
Low,31017,18.64,0.544,0.653,0.463
Medium,39323,44.08,0.572,0.634,0.469


In [0]:
%sql
SELECT COUNT(*) AS total_clean_rows FROM workspace.spotify_silver.clean_tracks;

total_clean_rows
80140
